In [ ]:

# ================================================================
# SupplyChainAI — 04 SUPPLIER RISK INTELLIGENCE
# Single-Cell Jupyter Notebook
# ================================================================
#
# FEATURES
#  1. Supplier data ingestion and validation
#  2. Supplier performance profiling
#  3. Delivery reliability analysis
#  4. Lead-time risk assessment
#  5. Product dependency / concentration analysis
#  6. Defect and quality risk analysis
#  7. Transparent weighted risk scoring
#  8. Simulated-label ML risk classification
#  9. Stratified model evaluation
# 10. Supplier risk segmentation
# 11. Risk drivers and recommended actions
# 12. Feature importance
# 13. Charts and business KPIs
# 14. CSV, JSON and model exports
#
# INSTALL
# pip install pandas numpy matplotlib scikit-learn joblib
#
# IMPORTANT
# This is a supplier risk prototype.
# If verified historical supplier failure outcomes are
# unavailable, ML labels are synthetic and must not
# be represented as real supplier failure predictions.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score
)

warnings.filterwarnings("ignore", category=FutureWarning)

# ================================================================
# 1. CONFIGURATION
# ================================================================

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

ROOT = (
    Path("..")
    if Path("../data").exists() and not Path("data").exists()
    else Path(".")
)

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = ROOT / "reports" / "supplier_risk"
MODEL_DIR = ROOT / "models" / "supplier_risk"

for directory in [
    PROCESSED_DIR,
    REPORT_DIR,
    MODEL_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

RISK_WEIGHTS = {
    "delivery": 0.30,
    "lead_time": 0.20,
    "quality": 0.25,
    "dependency": 0.15,
    "volatility": 0.10
}

print("=" * 75)
print("SUPPLYCHAINAI — SUPPLIER RISK INTELLIGENCE")
print("=" * 75)

# ================================================================
# 2. LOAD DATA
# ================================================================

def load_csv(paths, dataset_name, required=True):
    for path in paths:
        if path.exists():
            frame = pd.read_csv(path)
            print(
                f"Loaded {dataset_name}: "
                f"{path} ({len(frame):,} rows)"
            )
            return frame

    if required:
        raise FileNotFoundError(
            f"Missing {dataset_name} dataset. "
            f"Expected one of: {paths}"
        )

    print(f"Optional {dataset_name} dataset not found.")
    return None


suppliers = load_csv(
    [
        DATA_DIR / "suppliers.csv",
        PROCESSED_DIR / "supplier_analysis.csv"
    ],
    "suppliers"
)

products = load_csv(
    [DATA_DIR / "products.csv"],
    "products"
)

sales = load_csv(
    [
        PROCESSED_DIR / "sales_clean.csv",
        DATA_DIR / "sales.csv"
    ],
    "sales",
    required=False
)

# ================================================================
# 3. VALIDATE INPUTS
# ================================================================

def validate_columns(df, required, name):
    missing = sorted(set(required) - set(df.columns))

    if missing:
        raise ValueError(
            f"{name} missing required columns: {missing}"
        )


validate_columns(
    suppliers,
    [
        "supplier_id",
        "lead_time_days",
        "on_time_delivery_rate",
        "defect_rate"
    ],
    "suppliers"
)

validate_columns(
    products,
    ["product_id", "supplier_id", "unit_price"],
    "products"
)

if sales is not None:
    validate_columns(
        sales,
        ["product_id", "quantity_sold"],
        "sales"
    )

numeric_supplier_cols = [
    "lead_time_days",
    "on_time_delivery_rate",
    "defect_rate"
]

for col in numeric_supplier_cols:
    suppliers[col] = pd.to_numeric(
        suppliers[col],
        errors="coerce"
    )

products["unit_price"] = pd.to_numeric(
    products["unit_price"],
    errors="coerce"
)

suppliers = suppliers.dropna(
    subset=["supplier_id"]
).copy()

products = products.dropna(
    subset=["product_id", "supplier_id"]
).copy()

if suppliers["supplier_id"].duplicated().any():
    raise ValueError(
        "Supplier IDs must be unique."
    )

if products["product_id"].duplicated().any():
    raise ValueError(
        "Product IDs must be unique."
    )

if suppliers.empty or products.empty:
    raise ValueError(
        "Supplier and product datasets must not be empty."
    )

# Validate actual rates rather than silently clipping
# invalid business source records.

invalid_rates = (
    (suppliers["on_time_delivery_rate"].dropna() < 0).any()
    or
    (suppliers["on_time_delivery_rate"].dropna() > 1).any()
    or
    (suppliers["defect_rate"].dropna() < 0).any()
    or
    (suppliers["defect_rate"].dropna() > 1).any()
)

if invalid_rates:
    raise ValueError(
        "Supplier delivery and defect rates must be "
        "between 0 and 1."
    )

# ================================================================
# 4. DATA QUALITY REPORT
# ================================================================

quality_rows = []

for name, df in {
    "suppliers": suppliers,
    "products": products
}.items():
    quality_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(
            df.isna().sum().sum()
        ),
        "duplicate_rows": int(
            df.duplicated().sum()
        )
    })

quality_df = pd.DataFrame(quality_rows)

print("\nDATA QUALITY")
print(quality_df.to_string(index=False))

# ================================================================
# 5. SUPPLIER DEPENDENCY ANALYSIS
# ================================================================

product_dependency = (
    products.groupby(
        "supplier_id",
        as_index=False
    )
    .agg(
        product_count=("product_id", "nunique"),
        average_product_price=("unit_price", "mean")
    )
)

total_products = max(
    products["product_id"].nunique(),
    1
)

product_dependency["product_dependency_share"] = (
    product_dependency["product_count"]
    / total_products
)

# ================================================================
# 6. OPTIONAL SALES EXPOSURE
# ================================================================

if sales is not None:
    sales["quantity_sold"] = pd.to_numeric(
        sales["quantity_sold"],
        errors="coerce"
    )

    sales = sales.dropna(
        subset=["product_id", "quantity_sold"]
    )

    sales = sales[
        sales["quantity_sold"] >= 0
    ].copy()

    sales_exposure = sales[
        ["product_id", "quantity_sold"]
    ].merge(
        products[
            ["product_id", "supplier_id", "unit_price"]
        ],
        on="product_id",
        how="inner",
        validate="many_to_one"
    )

    sales_exposure["estimated_revenue"] = (
        sales_exposure["quantity_sold"]
        * sales_exposure["unit_price"]
    )

    supplier_revenue = (
        sales_exposure.groupby(
            "supplier_id",
            as_index=False
        )
        .agg(
            estimated_revenue=(
                "estimated_revenue",
                "sum"
            ),
            units_sold=(
                "quantity_sold",
                "sum"
            )
        )
    )

else:
    supplier_revenue = pd.DataFrame(
        columns=[
            "supplier_id",
            "estimated_revenue",
            "units_sold"
        ]
    )

# ================================================================
# 7. BUILD SUPPLIER INTELLIGENCE TABLE
# ================================================================

supplier_df = suppliers.merge(
    product_dependency,
    on="supplier_id",
    how="left",
    validate="one_to_one"
)

supplier_df = supplier_df.merge(
    supplier_revenue,
    on="supplier_id",
    how="left",
    validate="one_to_one"
)

supplier_df["product_count"] = (
    supplier_df["product_count"]
    .fillna(0)
    .astype(int)
)

supplier_df["product_dependency_share"] = (
    supplier_df["product_dependency_share"]
    .fillna(0)
)

supplier_df["estimated_revenue"] = (
    supplier_df["estimated_revenue"]
    .fillna(0)
)

supplier_df["units_sold"] = (
    supplier_df["units_sold"]
    .fillna(0)
)

total_revenue = supplier_df[
    "estimated_revenue"
].sum()

supplier_df["revenue_exposure_share"] = (
    supplier_df["estimated_revenue"]
    / total_revenue
    if total_revenue > 0
    else 0.0
)

# ================================================================
# 8. MISSING DATA HANDLING
# ================================================================

supplier_df["lead_time_days"] = (
    supplier_df["lead_time_days"]
    .fillna(
        supplier_df["lead_time_days"].median()
    )
)

if supplier_df["lead_time_days"].isna().any():
    supplier_df["lead_time_days"] = (
        supplier_df["lead_time_days"].fillna(7)
    )

supplier_df["lead_time_days"] = (
    supplier_df["lead_time_days"]
    .clip(lower=1)
)

supplier_df["on_time_delivery_rate"] = (
    supplier_df["on_time_delivery_rate"]
    .fillna(0.90)
)

supplier_df["defect_rate"] = (
    supplier_df["defect_rate"]
    .fillna(0.02)
)

# ================================================================
# 9. NORMALIZED RISK FEATURES
# ================================================================

def minmax_score(series):
    series = pd.to_numeric(
        series,
        errors="coerce"
    ).fillna(0)

    minimum = series.min()
    maximum = series.max()

    if maximum <= minimum:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (
        (series - minimum)
        / (maximum - minimum)
    )


# Higher value = higher risk

supplier_df["delivery_risk"] = (
    1 - supplier_df["on_time_delivery_rate"]
)

supplier_df["lead_time_risk"] = (
    minmax_score(
        supplier_df["lead_time_days"]
    )
)

supplier_df["quality_risk"] = (
    supplier_df["defect_rate"]
)

supplier_df["dependency_risk"] = (
    minmax_score(
        supplier_df["revenue_exposure_share"]
        if total_revenue > 0
        else supplier_df["product_dependency_share"]
    )
)

# Without lead-time history, supplier-specific
# volatility cannot be measured.
# This proxy uses a combination of lead time and
# late delivery exposure, not observed volatility.

supplier_df["volatility_proxy"] = (
    0.5 * supplier_df["lead_time_risk"]
    + 0.5 * supplier_df["delivery_risk"]
)

# ================================================================
# 10. WEIGHTED SUPPLIER RISK SCORE
# ================================================================

supplier_df["risk_score"] = 100 * (
    RISK_WEIGHTS["delivery"]
    * supplier_df["delivery_risk"]
    +
    RISK_WEIGHTS["lead_time"]
    * supplier_df["lead_time_risk"]
    +
    RISK_WEIGHTS["quality"]
    * supplier_df["quality_risk"]
    +
    RISK_WEIGHTS["dependency"]
    * supplier_df["dependency_risk"]
    +
    RISK_WEIGHTS["volatility"]
    * supplier_df["volatility_proxy"]
)

supplier_df["risk_score"] = (
    supplier_df["risk_score"]
    .clip(0, 100)
    .round(2)
)

# Thresholds are configurable heuristic categories.

supplier_df["risk_category"] = pd.cut(
    supplier_df["risk_score"],
    bins=[-0.001, 20, 40, 60, 100.001],
    labels=[
        "Low",
        "Medium",
        "High",
        "Critical"
    ],
    right=False
)

print("\nSUPPLIER RISK RANKING")

ranking_cols = [
    "supplier_id",
    "lead_time_days",
    "on_time_delivery_rate",
    "defect_rate",
    "product_count",
    "risk_score",
    "risk_category"
]

print(
    supplier_df[
        ranking_cols
    ].sort_values(
        "risk_score",
        ascending=False
    ).head(20).to_string(index=False)
)

# ================================================================
# 11. EXPLAIN RISK DRIVERS
# ================================================================

risk_components = {
    "Late deliveries": (
        supplier_df["delivery_risk"]
        * RISK_WEIGHTS["delivery"]
    ),
    "Long lead times": (
        supplier_df["lead_time_risk"]
        * RISK_WEIGHTS["lead_time"]
    ),
    "Quality defects": (
        supplier_df["quality_risk"]
        * RISK_WEIGHTS["quality"]
    ),
    "Supplier dependency": (
        supplier_df["dependency_risk"]
        * RISK_WEIGHTS["dependency"]
    ),
    "Volatility proxy": (
        supplier_df["volatility_proxy"]
        * RISK_WEIGHTS["volatility"]
    )
}

driver_df = pd.DataFrame(
    risk_components,
    index=supplier_df.index
)

supplier_df["primary_risk_driver"] = (
    driver_df.idxmax(axis=1)
)

def recommend_action(row):
    if row["risk_category"] == "Critical":
        return (
            "Escalate supplier review; evaluate backup "
            "suppliers and continuity plans"
        )

    if row["primary_risk_driver"] == "Late deliveries":
        return (
            "Review delivery SLAs and on-time performance"
        )

    if row["primary_risk_driver"] == "Quality defects":
        return (
            "Conduct supplier quality audit"
        )

    if row["primary_risk_driver"] == "Long lead times":
        return (
            "Review lead-time buffers and replenishment"
        )

    if row["primary_risk_driver"] == "Supplier dependency":
        return (
            "Evaluate alternate suppliers"
        )

    return (
        "Monitor performance and update supplier scorecard"
    )


supplier_df["recommended_action"] = (
    supplier_df.apply(
        recommend_action,
        axis=1
    )
)

# ================================================================
# 12. MACHINE LEARNING DATASET
# ================================================================

# No verified historical supplier failures are provided.
#
# A synthetic label is created only to demonstrate
# the ML workflow. Because it derives from the
# heuristic score, classifier performance is NOT
# evidence of actual supplier failure prediction.

ML_FEATURES = [
    "lead_time_days",
    "on_time_delivery_rate",
    "defect_rate",
    "product_count",
    "product_dependency_share",
    "revenue_exposure_share",
    "delivery_risk",
    "lead_time_risk",
    "quality_risk",
    "dependency_risk",
    "volatility_proxy"
]

supplier_df["synthetic_high_risk_label"] = (
    supplier_df["risk_score"] >= 40
).astype(int)

X = supplier_df[ML_FEATURES].fillna(0)
y = supplier_df["synthetic_high_risk_label"]

model = None
model_metrics = {
    "roc_auc": None,
    "average_precision": None,
    "test_samples": 0
}

# Require sufficient samples and both classes
# in train and test before attempting evaluation.

class_counts = y.value_counts()

can_train = (
    len(supplier_df) >= 20
    and y.nunique() == 2
    and class_counts.min() >= 4
)

if can_train:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    positive_idx = list(
        model.classes_
    ).index(1)

    probabilities = model.predict_proba(
        X_test
    )[:, positive_idx]

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    print("\nSIMULATED-LABEL CLASSIFICATION REPORT")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

    if y_test.nunique() == 2:
        model_metrics["roc_auc"] = float(
            roc_auc_score(
                y_test,
                probabilities
            )
        )

    model_metrics["average_precision"] = float(
        average_precision_score(
            y_test,
            probabilities
        )
    )

    model_metrics["test_samples"] = int(
        len(y_test)
    )

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "Lower Risk",
            "High Risk"
        ]
    ).plot(cmap="Blues")

    plt.title(
        "Supplier Risk Classifier — Synthetic Labels"
    )
    plt.tight_layout()

    plt.savefig(
        REPORT_DIR / "confusion_matrix.png"
    )
    plt.show()

    supplier_df["synthetic_ml_score"] = (
        model.predict_proba(X)[:, positive_idx]
    )

else:
    print(
        "\nML classifier skipped: insufficient suppliers "
        "or insufficient synthetic class diversity."
    )

    supplier_df["synthetic_ml_score"] = np.nan

# ================================================================
# 13. SUPPLIER RISK DISTRIBUTION
# ================================================================

risk_counts = (
    supplier_df["risk_category"]
    .value_counts()
    .reindex(
        ["Low", "Medium", "High", "Critical"],
        fill_value=0
    )
)

plt.figure(figsize=(10, 5))

plt.bar(
    risk_counts.index.astype(str),
    risk_counts.values
)

plt.title("Supplier Risk Distribution")
plt.xlabel("Risk Category")
plt.ylabel("Number of Suppliers")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "risk_distribution.png"
)
plt.show()

# ================================================================
# 14. DELIVERY RELIABILITY VS LEAD TIME
# ================================================================

plt.figure(figsize=(11, 6))

scatter = plt.scatter(
    supplier_df["lead_time_days"],
    supplier_df["on_time_delivery_rate"] * 100,
    c=supplier_df["risk_score"],
    cmap="viridis",
    s=80,
    alpha=0.85
)

plt.colorbar(
    scatter,
    label="Heuristic Risk Score"
)

plt.xlabel("Lead Time (Days)")
plt.ylabel("On-Time Delivery Rate (%)")
plt.title(
    "Supplier Delivery Reliability vs Lead Time"
)

plt.tight_layout()

plt.savefig(
    REPORT_DIR / "delivery_vs_lead_time.png"
)
plt.show()

# ================================================================
# 15. TOP 15 HIGH-RISK SUPPLIERS
# ================================================================

top_risk = (
    supplier_df.sort_values(
        "risk_score",
        ascending=False
    ).head(15)
)

plt.figure(figsize=(11, 7))

plt.barh(
    top_risk["supplier_id"].astype(str).iloc[::-1],
    top_risk["risk_score"].iloc[::-1]
)

plt.xlabel("Risk Score (0–100)")
plt.title("Top 15 Suppliers by Risk")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "top_risk_suppliers.png"
)
plt.show()

# ================================================================
# 16. SUPPLIER CONCENTRATION
# ================================================================

exposure_column = (
    "revenue_exposure_share"
    if total_revenue > 0
    else "product_dependency_share"
)

top_exposure = (
    supplier_df.sort_values(
        exposure_column,
        ascending=False
    ).head(10)
)

plt.figure(figsize=(11, 6))

plt.barh(
    top_exposure["supplier_id"].astype(str).iloc[::-1],
    (
        top_exposure[exposure_column] * 100
    ).iloc[::-1]
)

plt.xlabel("Supplier Exposure Share (%)")
plt.title("Top Supplier Dependencies")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "supplier_concentration.png"
)
plt.show()

# ================================================================
# 17. FEATURE IMPORTANCE
# ================================================================

if model is not None:
    feature_importance = pd.DataFrame({
        "feature": ML_FEATURES,
        "importance": model.feature_importances_
    }).sort_values(
        "importance",
        ascending=False
    )

    print("\nSIMULATED CLASSIFIER FEATURE IMPORTANCE")
    print(feature_importance.to_string(index=False))

    plt.figure(figsize=(11, 7))

    plt.barh(
        feature_importance["feature"].iloc[::-1],
        feature_importance["importance"].iloc[::-1]
    )

    plt.title(
        "Feature Importance — Synthetic Risk Labels"
    )

    plt.tight_layout()

    plt.savefig(
        REPORT_DIR / "feature_importance.png"
    )
    plt.show()

    feature_importance.to_csv(
        REPORT_DIR / "feature_importance.csv",
        index=False
    )

# ================================================================
# 18. BUSINESS KPIs
# ================================================================

high_risk_count = int(
    supplier_df["risk_category"]
    .isin(["High", "Critical"])
    .sum()
)

critical_count = int(
    (supplier_df["risk_category"] == "Critical").sum()
)

average_lead_time = float(
    supplier_df["lead_time_days"].mean()
)

average_delivery_rate = float(
    supplier_df["on_time_delivery_rate"].mean()
)

average_defect_rate = float(
    supplier_df["defect_rate"].mean()
)

top_supplier_share = float(
    supplier_df[exposure_column].max()
)

kpis = {
    "total_suppliers": int(len(supplier_df)),
    "high_or_critical_risk_suppliers": high_risk_count,
    "critical_risk_suppliers": critical_count,
    "average_risk_score": float(
        supplier_df["risk_score"].mean()
    ),
    "average_lead_time_days": average_lead_time,
    "average_on_time_delivery_rate": average_delivery_rate,
    "average_defect_rate": average_defect_rate,
    "largest_supplier_exposure_share": top_supplier_share
}

print("\nSUPPLIER RISK KPIs")
print(json.dumps(kpis, indent=2))

# ================================================================
# 19. EXPORT DATA
# ================================================================

supplier_df = supplier_df.sort_values(
    "risk_score",
    ascending=False
)

supplier_df.to_csv(
    PROCESSED_DIR / "supplier_risk_predictions.csv",
    index=False
)

supplier_df[
    supplier_df["risk_category"].isin(
        ["High", "Critical"]
    )
].to_csv(
    PROCESSED_DIR / "high_risk_suppliers.csv",
    index=False
)

quality_df.to_csv(
    REPORT_DIR / "data_quality.csv",
    index=False
)

if model is not None:
    model_artifact = {
        "model": model,
        "features": ML_FEATURES,
        "label_type": "synthetic_heuristic",
        "real_world_validated": False,
        "metrics": model_metrics
    }

    joblib.dump(
        model_artifact,
        MODEL_DIR / "supplier_risk_classifier.joblib"
    )

# ================================================================
# 20. JSON REPORT
# ================================================================

report = {
    "project": "SupplyChainAI",
    "notebook": "04_supplier_risk.ipynb",
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "risk_scoring_method": "weighted_heuristic",
    "risk_weights": RISK_WEIGHTS,
    "risk_thresholds": {
        "low": "0 to <20",
        "medium": "20 to <40",
        "high": "40 to <60",
        "critical": "60 to 100"
    },
    "kpis": kpis,
    "ml_model": (
        "RandomForestClassifier"
        if model is not None
        else None
    ),
    "ml_label_source": "synthetic_heuristic",
    "real_world_validated": False,
    "ml_test_metrics": model_metrics,
    "limitations": [
        "No historical supplier failure labels.",
        "ML labels are derived from heuristic risk scores.",
        "A random split is not a future-period validation.",
        "Lead-time volatility is approximated, not measured.",
        "Supplier concentration uses SKU count or estimated revenue.",
        "No supplier financial, geopolitical or compliance data.",
        "Risk scores are not calibrated failure probabilities.",
        "Recommendations require human review."
    ]
}

with open(
    REPORT_DIR / "supplier_risk_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        report,
        file,
        indent=2
    )

# ================================================================
# 21. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 75)
print("SUPPLYCHAINAI — SUPPLIER RISK ANALYSIS COMPLETED")
print("=" * 75)

print(f"Suppliers analyzed: {len(supplier_df)}")
print(f"High/Critical suppliers: {high_risk_count}")
print(f"Critical suppliers: {critical_count}")
print(f"Average lead time: {average_lead_time:.2f} days")
print(
    f"Average on-time delivery: "
    f"{average_delivery_rate * 100:.2f}%"
)

print("\nGenerated files:")
print("data/processed/supplier_risk_predictions.csv")
print("data/processed/high_risk_suppliers.csv")
print("reports/supplier_risk/supplier_risk_report.json")
print("reports/supplier_risk/risk_distribution.png")
print("reports/supplier_risk/delivery_vs_lead_time.png")
print("reports/supplier_risk/top_risk_suppliers.png")
print("reports/supplier_risk/supplier_concentration.png")

if model is not None:
    print("models/supplier_risk/supplier_risk_classifier.joblib")

print("\nNext notebook:")
print("05_inventory_optimization.ipynb")

print("=" * 75)
